# M12.5.2 — first X-only reference

CPU analysis after the user-run patch-MLP training and inference. No HDF5 reads, model loading, training or inference here. The primary contrast is M11 squared error minus reference squared error on identical validation observations, per label. Positive is favorable.

See `../../docs/experiments/M12_5_2_X_only_reference.md` for the fixed method, budget, contracts and commands. This is a point-estimator comparison, not a posterior or Bayes-risk estimate. Validation selects the checkpoint, so results and intervals are exploratory.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "src/paths.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.paths import resolve_data_root
from scripts.m12.analyze_m12_5_2_reference import analyze
from scripts.m12.run_m12_5_2_reference import digest, write_json

DATA_ROOT = resolve_data_root()  # Set CBC_PE_DATA_ROOT to the local external artifact root.
STUDY = DATA_ROOT / "results/M11_final_G1_500k/M12_5_2"
INFERENCE = STUDY / "patch_mlp_seed123_val_v1"
BOOTSTRAP = 2000


## Provenance and primary endpoint

The analysis verifies the receipt and scalar prediction archive digest, frozen M11 identity, complete validation row order, unique sources and finite arrays. Bootstrap resamples paired sources within fixed observed environment counts. Bonferroni intervals cover the family of three targets nominally; they do not correct checkpoint selection, cover new environments or estimate training-seed variation.


In [ ]:
endpoint, metrics, strata, paired = analyze(INFERENCE, bootstrap=BOOTSTRAP)
receipt = json.loads((INFERENCE / "complete.json").read_text())
display(pd.Series({"version": receipt["identity"]["version"],
                   "checkpoint_sha256": receipt["checkpoint_sha256"],
                   "baseline_sha256": receipt["baseline_sha256"],
                   "selected_epoch": receipt["epoch"], "split": receipt["split"]}))
display(endpoint)


## Bias and failure strata

Physical RMSE/MAE/R², prediction-on-truth OLS slopes and intercepts, and signed Q10/Q90 bias use the historical metric definitions and train-derived tail cutoffs. Region tables are descriptive: review chi_eff, low SNR, extreme peripherality, mass/spin cross-tails and every environment before proposing confirmation.


In [ ]:
display(metrics)
display(strata)
labels = ["chirp_mass", "total_mass", "chi_eff"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, label in zip(axes, labels):
    part = strata[(strata.label == label) & strata.stratum.str.startswith(("SNR_", "Mahalanobis_"))]
    ratio = np.sqrt(part.reference_mse_std.to_numpy() / part.m11_mse_std.to_numpy())
    ax.plot(np.arange(len(part)), ratio, "o")
    ax.axhline(1, color="black", linewidth=1)
    ax.axhline(1.02, color="red", linestyle="--", label="global guardrail (context only)")
    ax.set_xticks(np.arange(len(part)))
    ax.set_xticklabels(part.stratum, rotation=70, ha="right")
    ax.set_title(label)
axes[0].set_ylabel("Reference / M11 RMSE")
fig.tight_layout()


## Decision record — fill after reviewing results

- Per-target practical improvement requires the favorable-delta interval lower bound above 1% of M11 MSE. Equivalence requires the whole interval within ±that margin.
- Candidate for confirmation: at least one practical improvement, global RMSE ratio ≤1.02 for all targets, and a documented review of subgroup failures. The plotted 1.02 line is not a new subgroup acceptance test.
- Record training convergence/budget exhaustion from `history.json`, any concerns, and the proposed fixed checkpoint and confirmatory population. No automatic follow-on experiment.
- A negative result cannot establish an information limit. A positive result requires independent confirmation before a reproducibility claim. Risk of a validated point estimator bounds Bayes risk from above; it does not identify irreducible variance.

**Decision / unresolved concerns:** pending user-run results.


In [ ]:
# Optional scalar exports, disabled by default. Use a new directory for each analysis.
EXPORT = False
OUTPUT = STUDY / "patch_mlp_seed123_notebook_analysis_v1"
if EXPORT:
    OUTPUT = OUTPUT.resolve()
    if ROOT == OUTPUT or ROOT in OUTPUT.parents:
        raise ValueError("Exports must be outside Git")
    OUTPUT.mkdir(parents=True, exist_ok=False)
    tables = {"endpoint": endpoint, "metrics": metrics, "strata": strata, "paired_errors": paired}
    for name, table in tables.items():
        table.to_csv(OUTPUT / (name + ".csv"), index=False)
    write_json(OUTPUT / "analysis.json", {
        "bootstrap": BOOTSTRAP, "seed": 125202, "selection": "validation, exploratory",
        "inference_receipt_sha256": digest(INFERENCE / "complete.json"),
        "analysis_code_sha256": digest(ROOT / "scripts/m12/analyze_m12_5_2_reference.py"),
        "tables": {name: digest(OUTPUT / (name + ".csv")) for name in tables}})
